In [3]:
import numpy as np

i = np.arange(50)
x = 0.2 * i
y = 2.0 + 0.35*np.sin(0.7*x) + 0.15*np.cos(2.1*x) + 0.03*x

datos = np.column_stack([x, y])
np.savetxt("../datos/datos_sensor.csv", datos,
           delimiter=",", header="x,y", comments="", fmt="%.6f")

print("CSV creado con", len(x), "datos")
print("Primeros 3:")
print(datos[:3])
print("Últimos 3:")
print(datos[-3:])

CSV creado con 50 datos
Primeros 3:
[[0.         2.15      ]
 [0.2        2.19180343]
 [0.4        2.2088439 ]]
Últimos 3:
[[9.4        2.47872652]
 [9.6        2.47468253]
 [9.8        2.4610293 ]]


# Verificar el CSV

In [4]:
import numpy as np

# Leer el CSV como única fuente de información
data = np.loadtxt("../datos/datos_sensor.csv", delimiter=",", skiprows=1)
x = data[:, 0]
y = data[:, 1]

# Verificaciones
n_datos = len(x)
x_min, x_max = x[0], x[-1]
h_prom = np.mean(np.diff(x))
h_constante = np.allclose(np.diff(x), np.diff(x)[0])
nan_x = np.any(np.isnan(x))
nan_y = np.any(np.isnan(y))

print(f"Número de datos:     {n_datos}")
print(f"x va de:             {x_min} a {x_max}")
print(f"Paso h promedio:     {h_prom:.6f}")
print(f"Paso h constante:    {h_constante}")
print(f"¿NaN en x?:          {nan_x}")
print(f"¿NaN en y?:          {nan_y}")

Número de datos:     50
x va de:             0.0 a 9.8
Paso h promedio:     0.200000
Paso h constante:    True
¿NaN en x?:          False
¿NaN en y?:          False


# Parte D — Integración con Python y SciPy

In [2]:
import numpy as np
from scipy.integrate import trapezoid, simpson

# --- Leer el CSV (asumiendo que SOLO se tienen los datos) ---
data = np.loadtxt("../datos/datos_sensor.csv", delimiter=",", skiprows=1)
x = data[:, 0]
y = data[:, 1]

print(f"Número de datos: {len(x)}")
print(f"x va de {x[0]:.2f} a {x[-1]:.2f}")
print(f"Paso h: {x[1]-x[0]:.6f}")
print()

# --- Trapecio compuesto ---
I_trap = trapezoid(y, x)
print(f"Trapecio:          I = {I_trap:.10f}")

# --- Simpson compuesto ---
# 49 intervalos es impar. Simpson en los primeros 48 + trapecio en el último.
I_simp_48   = simpson(y[:-1], x=x[:-1])
I_trap_last = 0.5 * (y[-2] + y[-1]) * (x[-1] - x[-2])
I_simpson   = I_simp_48 + I_trap_last

print(f"Simpson:           I = {I_simpson:.10f}")
print()
print(f"|trapecio - simpson| = {abs(I_trap - I_simpson):.3e}")

Número de datos: 50
x va de 0.00 a 9.80
Paso h: 0.200000

Trapecio:          I = 21.1908459000
Simpson:           I = 21.1919523333

|trapecio - simpson| = 1.106e-03


# Observaciones
Comparando ambos resultados podemos ver que la diferencia trapecio–Simpson es del orden de $10^{-3}$, coherente con el distinto orden de error de cada método ($O(h^2)$ para trapecio, $O(h^4)$ para Simpson).


In [3]:
import pandas as pd

tabla = pd.DataFrame({
    'metodo': ['trapecio', 'simpson'],
    'integral': [I_trap, I_simpson]
})

tabla.to_csv("../resultados/ej2_python.csv", index=False)
print(tabla.to_string(index=False))

  metodo  integral
trapecio 21.190846
 simpson 21.191952


### Comparación entre entornos

| Método | Octave | C++ (GSL) | Python (SciPy) |
|---|---|---|---|
| Trapecio | 21.1908459000 | 21.1908459000 | 21.1908459000 |
| Simpson / spline | 21.1919523333 | ≈ 21.191978 | 21.1919523333 |
| Diferencia trap-simp | 1.106e-03 | ≈ 1.132e-03 | 1.106e-03 |

**Conclusión:** los tres entornos producen resultados equivalentes. Las diferencias son del orden de $10^{-3}$, coherentes con la diferencia de orden entre trapecio y Simpson/spline.